In [2]:
import faiss

index = faiss.read_index(
    "vector_store/cfpb_faiss.index"
)

print("Total vectors:", index.ntotal)

Total vectors: 2098


In [3]:
import os

print(os.getcwd())

C:\Users\Soret\rag-complaint-chatbo


In [4]:
os.listdir()

['.git',
 '.github',
 '.gitignore',
 '.ipynb_checkpoints',
 '.vscode',
 'app.py',
 'data',
 'notebooks',
 'RAGB.ipynb',
 'README.md',
 'requirements.txt',
 'src',
 'tests',
 'Text Chunking.ipynb',
 'vector_store',
 'venv']

In [5]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if "meta" in file.lower() or "faiss" in file.lower():
            print(os.path.join(root, file))

.\vector_store\cfpb_faiss.index
.\vector_store\cfpb_metadata.pkl
.\venv\Lib\site-packages\aiohappyeyeballs-2.6.2.dist-info\METADATA
.\venv\Lib\site-packages\aiohttp-3.14.1.dist-info\METADATA
.\venv\Lib\site-packages\aiosignal-1.4.0.dist-info\METADATA
.\venv\Lib\site-packages\annotated_doc-0.0.4.dist-info\METADATA
.\venv\Lib\site-packages\annotated_types-0.7.0.dist-info\METADATA
.\venv\Lib\site-packages\anyio-4.14.0.dist-info\METADATA
.\venv\Lib\site-packages\argon2_cffi-25.1.0.dist-info\METADATA
.\venv\Lib\site-packages\argon2_cffi_bindings-25.1.0.dist-info\METADATA
.\venv\Lib\site-packages\arrow-1.4.0.dist-info\METADATA
.\venv\Lib\site-packages\asttokens-3.0.1.dist-info\METADATA
.\venv\Lib\site-packages\async_lru-2.3.0.dist-info\METADATA
.\venv\Lib\site-packages\attrs-26.1.0.dist-info\METADATA
.\venv\Lib\site-packages\babel-2.18.0.dist-info\METADATA
.\venv\Lib\site-packages\bcrypt-5.0.0.dist-info\METADATA
.\venv\Lib\site-packages\beautifulsoup4-4.15.0.dist-info\METADATA
.\venv\Lib\sit

In [5]:
import os

print(os.listdir("vector_store"))

['cfpb_faiss.index', 'cfpb_metadata.pkl']


In [6]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles in current directory:")
print(os.listdir())

Current directory:
C:\Users\Soret\rag-complaint-chatbo

Files in current directory:
['.git', '.github', '.gitignore', '.ipynb_checkpoints', '.vscode', 'app.py', 'data', 'notebooks', 'RAGB.ipynb', 'README.md', 'requirements.txt', 'src', 'tests', 'Text Chunking.ipynb', 'vector_store', 'venv']


In [7]:
import faiss
import pickle

index = faiss.read_index(
    r"vector_store\cfpb_faiss.index"
)

with open(
    r"vector_store\cfpb_metadata.pkl",
    "rb"
) as f:
    metadata = pickle.load(f)

print("Vectors:", index.ntotal)
print("Metadata:", len(metadata))

Vectors: 2098
Metadata: 2098


In [8]:
import numpy as np

def embed_question(question):
    """
    Convert a user question into a vector embedding.
    """

    embedding = embedding_model.encode(
        question,
        convert_to_numpy=True
    )

    return embedding.astype("float32")

In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully


In [10]:
query_vector = embed_question(
    "Why was my credit card charged without my permission?"
)

print(query_vector.shape)

(384,)


In [11]:
def retrieve_chunks(
    question,
    k=5
):
    """
    Retrieve top-k most relevant complaint chunks.
    """

    query_embedding = embed_question(question)

    query_embedding = np.array(
        [query_embedding],
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        results.append({
            "rank": rank + 1,
            "score": float(distances[0][rank]),
            "metadata": metadata[idx]
        })

    return results

In [12]:
results = retrieve_chunks(
    "Unauthorized transactions on my credit card",
    k=5
)

for r in results:

    print(f"\nRank: {r['rank']}")
    print(f"Score: {r['score']}")
    print(r["metadata"])


Rank: 1
Score: 0.6174334287643433
{'complaint_id': '13968411', 'product': 'Credit card', 'chunk_id': 6, 'text': 'ofxx xx xxxxdollars i also request that you investigate these unauthorized transaction and take steps to prevent future fraudulent charges to my account n ni have attached copies of my credit card statement and other documents that support my claim n nplease send me written confirmation that you have removed the fraudulent charges and credited my account and also that you have investigated the matter n nthank you for your prompt attention to this matter'}

Rank: 2
Score: 0.7186909914016724
{'complaint_id': '13941218', 'product': 'Credit card', 'chunk_id': 1, 'text': 'card has had a 0 00 balance for years so i logged into my account and discovered a new unauthorized transaction for 1900 00 i immediately contacted citibank to report it as fraud this is case xxxx according to their records while following up with xxxx xxxx and citis fraud teams i was told the purchase occurred

In [13]:
print(metadata[0])

{'complaint_id': '14060912', 'product': 'Credit card', 'chunk_id': 0, 'text': 'we called citi bank customer service on xx xx year to question a late fee 30 00 and an interest charge on our account 3 00 as we always pay on time the bill was due xx xx year on a saturday and we were told the payment was on xx xx year the sunday after we asked to be given an adjustment of those financial charges but were declined instead our account was closed after a heated conversation and the manager did not disclose to us what was going to happen to our accumulated cash back rewards of 26'}


In [14]:
results = retrieve_chunks(
    "Someone used my credit card without permission"
)

for r in results:

    item = r["metadata"]

    print("="*80)
    print("complaint_id:", item["complaint_id"])
    print("product:", item["product"])
    print("Similarity Score:", r["score"])

    print("\nChunk:")
    print(item["text"][:500])

complaint_id: 13769765
product: Credit card
Similarity Score: 0.6124476194381714

Chunk:
charges made with my credit card i became alarmed that this activity could still be going on i was greeted by a bank representative that was very unfriendly after i gave her my card number social security number my security name i proceeded to tell her that my credit card had been stolen she then informed me that because i had the audacity to get a new cell phone this past year she would not help me or close the account i explained how ridiculous that was considering that i had the account for
complaint_id: 13697351
product: Credit card
Similarity Score: 0.6299588680267334

Chunk:
there was nothing on my account that showed i had purchased or paid for anything in that amount to xxxx she informed me to call my credit card company and dispute the charges and change my credit card because someone had used it without my authorization i then called citi visa and explained to customer service what happen

In [15]:
print(item["complaint_id"])
print(item["product"])
print(item["text"])

13697351
Credit card
card number on file as my bill is paid through my checking account i did however purchase a phone from xxxx on xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx and used my credit card to pay for the phone xxxx xxxxold me to ask visa what additional information they would need from xxxx to back up that the item purchased was done without my permission and not on my account i called visa back and was informed that the dispute was closed and there was nothing they could do about it they said i was an


In [22]:
dir()

['AutoModelForSeq2SeqLM',
 'AutoTokenizer',
 'In',
 'Out',
 'SentenceTransformer',
 '_',
 '_9',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__session__',
 '__spec__',
 '_dh',
 '_i',
 '_i1',
 '_i10',
 '_i11',
 '_i12',
 '_i13',
 '_i14',
 '_i15',
 '_i16',
 '_i17',
 '_i18',
 '_i19',
 '_i2',
 '_i20',
 '_i21',
 '_i22',
 '_i3',
 '_i4',
 '_i5',
 '_i6',
 '_i7',
 '_i8',
 '_i9',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'embed_question',
 'embedding_model',
 'evaluation_questions',
 'exit',
 'f',
 'faiss',
 'generate_answer',
 'get_ipython',
 'index',
 'inputs',
 'metadata',
 'model',
 'model_name',
 'np',
 'open',
 'outputs',
 'pickle',
 'prompt',
 'q',
 'question',
 'quit',
 'r',
 'rag_pipeline',
 'results',
 'retrieve_chunks',
 'test_prompt',
 'tokenizer']

In [23]:
question = "What are the most common complaints?"

retrieved_chunks = retrieve_chunks(question)

print(retrieved_chunks)

[{'complaint_id': '13970077', 'product': 'Checking or savings account', 'chunk_id': 11, 'text': 'we appreciate your participation in the complaint process and your feedback on the companys response both are important to us and consumers who may have similar issues and concerns'}, {'complaint_id': '12705137', 'product': 'Money transfer, virtual currency, or money service', 'chunk_id': 1, 'text': 'so me being a consumer i was totally being ignored and no one never even extended to investigate or follow up on my complaint enough is enough this needs to stop xxxxxxxx xxxx xxxx xxxx xxxx xxxxand to be treated like xxxx'}, {'complaint_id': '13805562', 'product': 'Payday loan, title loan, personal loan, or advance loan', 'chunk_id': 3, 'text': 'charges the billing practices the misrepresentation of the services and the failure to accommodate serious medical concerns'}, {'complaint_id': '13406106', 'product': 'Checking or savings account', 'chunk_id': 0, 'text': 'formal complaint regarding ong

In [25]:
print(retrieved_chunks[0])

{'complaint_id': '13970077', 'product': 'Checking or savings account', 'chunk_id': 11, 'text': 'we appreciate your participation in the complaint process and your feedback on the companys response both are important to us and consumers who may have similar issues and concerns'}


In [26]:
context = "\n\n".join(chunk["text"] for chunk in retrieved_chunks)

In [27]:
print(retrieved_chunks[0].keys())

dict_keys(['complaint_id', 'product', 'chunk_id', 'text'])


In [28]:
import inspect

print(inspect.getsource(retrieve_chunks))

def retrieve_chunks(question, k=5):

    query_embedding = embed_question(question)

    query_embedding = np.array(
        [query_embedding],
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for idx in indices[0]:
        results.append(metadata[idx])

    return results



In [29]:
rag_pipeline

<function __main__.rag_pipeline(question, k=5)>

In [31]:
prompt_template = """
You are a helpful assistant.

Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

In [32]:
response = rag_pipeline("What are the common complaints?")
print(response)

(iii)


In [33]:
%whos

Variable                Type                          Data/Info
---------------------------------------------------------------
AutoModelForSeq2SeqLM   type                          <class 'transformers.mode<...>o.AutoModelForSeq2SeqLM'>
AutoTokenizer           type                          <class 'transformers.mode<...>tion_auto.AutoTokenizer'>
SentenceTransformer     ABCMeta                       <class 'sentence_transfor<...>del.SentenceTransformer'>
context                 str                           we appreciate your partic<...> feedback on the companys
embed_question          function                      <function embed_question at 0x000001D762AA6350>
embedding_model         SentenceTransformer           SentenceTransformer(\n  (<...>\n  (2): Normalize({})\n)
evaluation_questions    list                          n=8
f                       BufferedReader                <_io.BufferedReader name=<...>store/cfpb_metadata.pkl'>
faiss                   module                      

In [34]:
prompt_template = """
Use the context below to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

def rag_pipeline(question, k=3):
    retrieved_chunks = retrieve_chunks(question, k)

    context = "\n\n".join(
        chunk["text"] for chunk in retrieved_chunks
    )

    prompt = prompt_template.format(
        context=context,
        question=question
    )

    answer = generate_answer(prompt)

    return answer

In [35]:
import inspect
print(inspect.getsource(rag_pipeline))

def rag_pipeline(question, k=3):
    retrieved_chunks = retrieve_chunks(question, k)

    context = "\n\n".join(
        chunk["text"] for chunk in retrieved_chunks
    )

    prompt = prompt_template.format(
        context=context,
        question=question
    )

    answer = generate_answer(prompt)

    return answer



In [22]:
PROMPT_TEMPLATE = """
You are a financial analyst assistant for CrediTrust specializing in customer complaint analysis.

Your responsibility is to answer questions strictly based on the retrieved complaint excerpts.

Retrieved Complaint Excerpts:
{context}

User Question:
{question}

Requirements:
- Use only the information found in the complaint excerpts.
- Do not rely on prior knowledge.
- Cite relevant complaint themes mentioned in the context.
- When appropriate, explain the evidence supporting your answer.
- If the retrieved complaints do not provide sufficient information, explicitly state:
  "I do not have enough information in the retrieved complaints to answer this question."
- Never invent facts, customer experiences, or company actions.
- Provide a clear and professional response.

Answer:
"""

In [16]:
def build_prompt(context, question):
    return PROMPT_TEMPLATE.format(
        context=context,
        question=question
    )

In [21]:
import transformers
print(transformers.__version__)

5.12.1


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

def generate_answer(prompt):
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
PROMPT_TEMPLATE = """
You are a financial analyst assistant for CrediTrust.

Your task is to answer questions about customer complaints using only the information contained in the retrieved complaint excerpts.

Instructions:
- Use only the provided context.
- Do not make assumptions.
- If the answer is not in the context, say:
  "I do not have enough information in the retrieved complaints to answer this question."
- Be concise and professional.

Context:
{context}

Question:
{question}

Answer:
"""

In [8]:
def generate_answer(question, retrieved_chunks):

    context = "\n\n".join(retrieved_chunks)

    prompt = PROMPT_TEMPLATE.format(
        context=context,
        question=question
    )

    response = llm(prompt)

    answer = response[0]["generated_text"]

    return answer

In [9]:
def rag_pipeline(question, k=5):

    query_embedding = embed_question(question)

    retrieved_chunks = retrieve_chunks(
        query_embedding,
        k=k
    )

    answer = generate_answer(
        question,
        retrieved_chunks
    )

    return answer

In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [32]:
print(type(query_embedding))

<class 'numpy.ndarray'>


In [11]:
import numpy as np

def retrieve_chunks(query_embedding, k=5):
    """
    Retrieve top-k complaint chunks from FAISS.
    """

    query_embedding = np.array(
        [query_embedding],
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = [
        metadata[i]
        for i in indices[0]
    ]

    return results

In [35]:
question = "Why are customers complaining about unauthorized credit card charges?"

query_embedding = embed_question(question)

results = retrieve_chunks(
    query_embedding,
    k=5
)

print(results[:2])

[{'complaint_id': '13816574', 'product': 'Credit card', 'chunk_id': 0, 'text': 'this company is using misleading tacticts deceving all these fees the credit limit was very low as 500 00 they did not disclose the annual fee of 120 00 will be charged within 5 months after the card was issued the fee is almost 30 of the credit limit and they will charge to added fees the interest rate is very high at 36 00 and the balance goes up to the point that its impossible will be not able to pay off i am an xxxx xxxx'}, {'complaint_id': '13941218', 'product': 'Credit card', 'chunk_id': 3, 'text': 'it was not fraud and are rebilling the charge to me why im filing this complaint citibank has failed to properly investigate this fraud claim and is holding me liable for a charge i clearly did not authorize or benefit from the card has not been activated or used physically and i reported the fraud immediately upon discovery the cardholder agreement outlines protections against unauthorized charges and i 

In [17]:
def retrieve_chunks(question, k=5):

    query_embedding = embed_question(question)

    query_embedding = np.array(
        [query_embedding],
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    return [metadata[i] for i in indices[0]]

In [19]:
def embed_question(question):
    """
    Convert a user question into an embedding vector.
    """

    embedding = embedding_model.encode(
        str(question),
        convert_to_numpy=True
    )

    return embedding

In [20]:
def retrieve_chunks(query_embedding, k=5):

    query_embedding = np.array(
        [query_embedding],
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    return [metadata[i] for i in indices[0]]

In [21]:
def rag_pipeline(question, k=5):

    query_embedding = embed_question(question)

    retrieved_chunks = retrieve_chunks(
        query_embedding,
        k=k
    )

    return retrieved_chunks

In [41]:
import inspect

print(inspect.getsource(retrieve_chunks))

def retrieve_chunks(query_embedding, k=5):

    query_embedding = np.array(
        [query_embedding],
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    return [metadata[i] for i in indices[0]]



In [22]:
def build_context(results):

    context_parts = []

    for item in results:

        context_parts.append(
            f"""
Product: {item['product']}
Complaint ID: {item['complaint_id']}
Complaint Text:
{item['text']}
"""
        )

    return "\n\n".join(context_parts)

In [23]:
import faiss
import pickle

index = faiss.read_index(
    "vector_store/cfpb_faiss.index"
)

with open(
    "vector_store/cfpb_metadata.pkl",
    "rb"
) as f:
    metadata = pickle.load(f)

In [25]:
import numpy as np

def retrieve_chunks(question, k=5):

    query_embedding = embed_question(question)

    query_embedding = np.array(
        [query_embedding],
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for idx in indices[0]:
        results.append(metadata[idx])

    return results

In [26]:
results = retrieve_chunks(
    "Why are customers complaining about unauthorized credit card charges?"
)

print(results[0])

{'complaint_id': '13816574', 'product': 'Credit card', 'chunk_id': 0, 'text': 'this company is using misleading tacticts deceving all these fees the credit limit was very low as 500 00 they did not disclose the annual fee of 120 00 will be charged within 5 months after the card was issued the fee is almost 30 of the credit limit and they will charge to added fees the interest rate is very high at 36 00 and the balance goes up to the point that its impossible will be not able to pay off i am an xxxx xxxx'}


In [27]:
print("embed_question" in globals())
print("embedding_model" in globals())
print("index" in globals())
print("metadata" in globals())

True
True
True
True


In [32]:
results = retrieve_chunks(
    "Why are customers complaining about unauthorized credit card charges?"
)

for i, chunk in enumerate(results, start=1):
    print(f"\nChunk {i}")
    print(chunk)


Chunk 1
{'complaint_id': '13816574', 'product': 'Credit card', 'chunk_id': 0, 'text': 'this company is using misleading tacticts deceving all these fees the credit limit was very low as 500 00 they did not disclose the annual fee of 120 00 will be charged within 5 months after the card was issued the fee is almost 30 of the credit limit and they will charge to added fees the interest rate is very high at 36 00 and the balance goes up to the point that its impossible will be not able to pay off i am an xxxx xxxx'}

Chunk 2
{'complaint_id': '13941218', 'product': 'Credit card', 'chunk_id': 3, 'text': 'it was not fraud and are rebilling the charge to me why im filing this complaint citibank has failed to properly investigate this fraud claim and is holding me liable for a charge i clearly did not authorize or benefit from the card has not been activated or used physically and i reported the fraud immediately upon discovery the cardholder agreement outlines protections against unauthorize

In [36]:
retrieved_chunks = retrieve_chunks(
    "Why are customers complaining about unauthorized credit card charges?"
)

context = "\n\n".join(
    [chunk["text"] for chunk in retrieved_chunks]
)

print(context[:1000])

this company is using misleading tacticts deceving all these fees the credit limit was very low as 500 00 they did not disclose the annual fee of 120 00 will be charged within 5 months after the card was issued the fee is almost 30 of the credit limit and they will charge to added fees the interest rate is very high at 36 00 and the balance goes up to the point that its impossible will be not able to pay off i am an xxxx xxxx

it was not fraud and are rebilling the charge to me why im filing this complaint citibank has failed to properly investigate this fraud claim and is holding me liable for a charge i clearly did not authorize or benefit from the card has not been activated or used physically and i reported the fraud immediately upon discovery the cardholder agreement outlines protections against unauthorized charges and i believe citi is violating those terms additionally xxxx xxxx enabled this fraud by not

i recieved a pre approved card in the mail i activated the card the card 

In [37]:
print(type(context))
print(type(question))

<class 'str'>
<class 'str'>


In [38]:
prompt_template = """
You are a financial analyst assistant for CrediTrust.

Your task is to answer questions about customer complaints.

Use ONLY the information provided in the retrieved complaint excerpts.

If the retrieved context does not contain enough information to answer the question, say:
"I do not have enough information in the provided complaint data to answer this question."

Provide a concise and factual answer.

Context:
{context}

Question:
{question}

Answer:
"""

In [39]:
prompt = prompt_template.format(
    context=context,
    question=question
)

print(prompt[:1500])


You are a financial analyst assistant for CrediTrust.

Your task is to answer questions about customer complaints.

Use ONLY the information provided in the retrieved complaint excerpts.

If the retrieved context does not contain enough information to answer the question, say:
"I do not have enough information in the provided complaint data to answer this question."

Provide a concise and factual answer.

Context:
this company is using misleading tacticts deceving all these fees the credit limit was very low as 500 00 they did not disclose the annual fee of 120 00 will be charged within 5 months after the card was issued the fee is almost 30 of the credit limit and they will charge to added fees the interest rate is very high at 36 00 and the balance goes up to the point that its impossible will be not able to pay off i am an xxxx xxxx

it was not fraud and are rebilling the charge to me why im filing this complaint citibank has failed to properly investigate this fraud claim and is h

In [40]:
import transformers
print(transformers.__version__)

5.12.1


In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [2]:
print("Model loaded successfully")

Model loaded successfully


In [4]:
prompt = "Answer: Why are customers complaining about unauthorized credit card charges?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

unauthorized credit card charges


In [5]:
def generate_answer(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [6]:
test_prompt = """
Context:
Customers reported unauthorized charges on their credit cards.
Several customers stated transactions appeared that they did not make.

Question:
Why are customers complaining about unauthorized credit card charges?

Answer:
"""

print(generate_answer(test_prompt))

a).


In [7]:
def rag_pipeline(question, k=5):

    retrieved_chunks = retrieve_chunks(
        question,
        k=k
    )

    context = "\n\n".join(
        [chunk["text"] for chunk in retrieved_chunks]
    )

    prompt = prompt_template.format(
        context=context,
        question=question
    )

    answer = generate_answer(prompt)

    return answer

In [10]:
import faiss
import pickle
import numpy as np

from sentence_transformers import SentenceTransformer

In [11]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
index = faiss.read_index(
    "vector_store/cfpb_faiss.index"
)

In [13]:
with open(
    "vector_store/cfpb_metadata.pkl",
    "rb"
) as f:
    metadata = pickle.load(f)

In [14]:
print(type(metadata))
print(len(metadata))
print(metadata[0])

<class 'list'>
2098
{'complaint_id': '14060912', 'product': 'Credit card', 'chunk_id': 0, 'text': 'we called citi bank customer service on xx xx year to question a late fee 30 00 and an interest charge on our account 3 00 as we always pay on time the bill was due xx xx year on a saturday and we were told the payment was on xx xx year the sunday after we asked to be given an adjustment of those financial charges but were declined instead our account was closed after a heated conversation and the manager did not disclose to us what was going to happen to our accumulated cash back rewards of 26'}


In [15]:
def embed_question(question):

    return embedding_model.encode(
        str(question),
        convert_to_numpy=True
    )

In [16]:
def retrieve_chunks(question, k=5):

    query_embedding = embed_question(question)

    query_embedding = np.array(
        [query_embedding],
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for idx in indices[0]:
        results.append(metadata[idx])

    return results

In [17]:
results = retrieve_chunks(
    "Why are customers complaining about unauthorized credit card charges?"
)

print(results[0])

{'complaint_id': '13816574', 'product': 'Credit card', 'chunk_id': 0, 'text': 'this company is using misleading tacticts deceving all these fees the credit limit was very low as 500 00 they did not disclose the annual fee of 120 00 will be charged within 5 months after the card was issued the fee is almost 30 of the credit limit and they will charge to added fees the interest rate is very high at 36 00 and the balance goes up to the point that its impossible will be not able to pay off i am an xxxx xxxx'}


In [18]:
for r in results[:2]:
    print("\nPRODUCT:", r["product"])
    print("\nTEXT:")
    print(r["text"][:500])
    print("-" * 80)


PRODUCT: Credit card

TEXT:
this company is using misleading tacticts deceving all these fees the credit limit was very low as 500 00 they did not disclose the annual fee of 120 00 will be charged within 5 months after the card was issued the fee is almost 30 of the credit limit and they will charge to added fees the interest rate is very high at 36 00 and the balance goes up to the point that its impossible will be not able to pay off i am an xxxx xxxx
--------------------------------------------------------------------------------

PRODUCT: Credit card

TEXT:
it was not fraud and are rebilling the charge to me why im filing this complaint citibank has failed to properly investigate this fraud claim and is holding me liable for a charge i clearly did not authorize or benefit from the card has not been activated or used physically and i reported the fraud immediately upon discovery the cardholder agreement outlines protections against unauthorized charges and i believe citi is violati

In [37]:
RAG_PROMPT_TEMPLATE = """
You are a consumer compliance analyst. Answer the question using ONLY the context provided below.

CRITICAL RESTRAINTS:
- Do NOT under any circumstances output standalone symbols, letters (e.g., 'a).'), or Roman numerals (e.g., '(ii)').
- You must write out your analytical findings in full, complete, descriptive sentences.
- Do not copy formatting choices from the source text; extract the factual information.

Context: {context}
Question: {question}

Analytical Answer:
"""

In [38]:
generation_config = {
    "max_new_tokens": 256,       # Ensure the model has enough room to talk
    "temperature": 0.1,          # Low temperature keeps it strictly factual
    "do_sample": False           # Deterministic execution reduces random token loops
}

In [39]:
evaluation_questions = [
    "What are the most common customer complaints?",
    "What billing issues are reported?",
    "What delivery problems do customers mention?",
    "What complaints relate to customer service?",
    "What refund issues are reported?"
]

for q in evaluation_questions:
    answer = rag_pipeline(q)
    chunks = retrieve_chunks(q)

    print("\nQUESTION:", q)
    print("\nANSWER:", answer)
    print("\nTOP SOURCES:")
    for c in chunks[:2]:
        print(c["text"][:200])


QUESTION: What are the most common customer complaints?

ANSWER: (ii).

TOP SOURCES:
we appreciate your participation in the complaint process and your feedback on the companys response both are important to us and consumers who may have similar issues and concerns
so me being a consumer i was totally being ignored and no one never even extended to investigate or follow up on my complaint enough is enough this needs to stop xxxxxxxx xxxx xxxx xxxx xxxx xxxxand t

QUESTION: What billing issues are reported?

ANSWER: a).

TOP SOURCES:
charges the billing practices the misrepresentation of the services and the failure to accommodate serious medical concerns
that the consumer financial protection bureau initiate a formal investigation into citibank macys billing systems and late fee practices examine whether such practices are disproportionately affecting

QUESTION: What delivery problems do customers mention?

ANSWER: (iii)

TOP SOURCES:
is unacceptable in 2025 and is clearly intended to

In [40]:
import numpy as np

def retrieve_chunks(question, k=5):

    query_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True
    )

    query_embedding = np.array(
        [query_embedding],
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for idx in indices[0]:
        results.append(metadata[idx])

    return results

In [46]:
print(type(q))
print(type(embed_question(q)))

<class 'str'>
<class 'numpy.ndarray'>


In [41]:
import numpy as np

def retrieve_chunks(question, k=5):

    query_embedding = embedding_model.encode(
        [question],          # list containing one string
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        k
    )

    return [metadata[i] for i in indices[0]]

In [42]:
def rag_pipeline(question, k=5):

    retrieved_chunks = retrieve_chunks(
        question,
        k=k
    )

    context = "\n\n".join(
        [chunk["chunk_text"] for chunk in retrieved_chunks]
    )

    prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = generator(
        prompt,
        max_new_tokens=200
    )

    return response[0]["generated_text"]

In [46]:
import gradio as gr

# Function that calls your RAG pipeline
def answer_question(question):
    try:
        answer = rag_pipeline(question)
        return answer
    except Exception as e:
        return f"Error: {str(e)}"

# Build interface
with gr.Blocks(title="Complaint RAG Chatbot") as demo:

    gr.Markdown("# Complaint RAG Chatbot")
    gr.Markdown("Ask questions about the complaint dataset.")

    question_input = gr.Textbox(
        label="Your Question",
        placeholder="Enter your question here..."
    )

    submit_btn = gr.Button("Ask")

    answer_output = gr.Textbox(
        label="AI Generated Answer",
        lines=8
    )

    submit_btn.click(
        fn=answer_question,
        inputs=question_input,
        outputs=answer_output
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


C:\Users\Soret\rag-complaint-chatbo\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Soret\rag-complaint-chatbo\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [47]:
import time

def answer_question_stream(question):

    retrieved_chunks = retrieve_chunks(question)

    context = "\n\n".join(
        chunk["text"] for chunk in retrieved_chunks
    )

    prompt = prompt_template.format(
        context=context,
        question=question
    )

    answer = generate_answer(prompt)

    sources = "\n\n".join(
        chunk["text"]
        for chunk in retrieved_chunks[:3]
    )

    partial = ""

    for word in answer.split():
        partial += word + " "
        time.sleep(0.03)
        yield partial, sources